In [ ]:
!pip install -U langchain langchain-community langchain-openai beautifulsoup4

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.12.3
    Uninstalling beautifulsoup4-4.12.3:
      Successfully uninstalled beautifulsoup4-4.12.3



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_community.document_loaders import WebBaseLoader

# ওয়েবসাইট লিংক দিন
url = "https://betopiagroup.com/"
loader = WebBaseLoader(url)

# ডাটা লোড করা
data = loader.load()

print(f"Website content loaded! Character count: {len(data[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Website content loaded! Character count: 4193


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 6


In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(chunks, embeddings)

print("Vector database is ready with website data!")

Vector database is ready with website data!


In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LLM এবং প্রম্পট সেটআপ
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# রিট্রিভার এবং চেইন তৈরি
retriever = vector_db.as_retriever()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Website QA Chain is Ready!")

Website QA Chain is Ready!


In [9]:
query = "What Betopia Group Core beliefs?"
response = rag_chain.invoke(query)

print("Answer:", response)

Answer: The Betopia Group core beliefs are:
1. Belief in the boundless potential of humanity
2. Empowering individuals and businesses to unleash their brilliance
3. Belief in achieving success together through collaboration
4. Belief that true prosperity is achieved when unlocking possibilities and moving forward limitless, together
